In [42]:
import pandas as pd


In [43]:
traffic_df = pd.read_csv("../data/raw/delhi_traffic_features.csv")

In [44]:
traffic_df.shape

(4000, 10)

In [45]:
traffic_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Trip_ID                4000 non-null   str    
 1   start_area             4000 non-null   str    
 2   end_area               4000 non-null   str    
 3   distance_km            4000 non-null   float64
 4   time_of_day            4000 non-null   str    
 5   day_of_week            4000 non-null   str    
 6   weather_condition      4000 non-null   str    
 7   traffic_density_level  4000 non-null   str    
 8   road_type              4000 non-null   str    
 9   average_speed_kmph     4000 non-null   float64
dtypes: float64(2), str(8)
memory usage: 312.6 KB


In [46]:
traffic_df.duplicated().sum()

np.int64(0)

In [47]:
for col in [
    "time_of_day",
    "day_of_week",
    "weather_condition",
    "traffic_density_level"
]:
    print(f"\n{col}")
    print(traffic_df[col].value_counts())


time_of_day
time_of_day
Evening Peak    1397
Morning Peak    1150
Afternoon       1057
Night            396
Name: count, dtype: int64

day_of_week
day_of_week
Weekday    3031
Weekend     969
Name: count, dtype: int64

weather_condition
weather_condition
Clear       2382
Heatwave     608
Rain         599
Fog          411
Name: count, dtype: int64

traffic_density_level
traffic_density_level
High         1348
Medium       1055
Very High     979
Low           618
Name: count, dtype: int64


In [48]:
traffic_df.groupby("traffic_density_level")["average_speed_kmph"].agg(
    ["count", "mean", "median", "min", "max"]
)

,count,mean,median,min,max
traffic_density_level,,,,,
High,1348,24.208012,23.05,7.6,46.2
Low,618,46.949515,43.55,10.2,93.3
Medium,1055,36.535450,35.10,7.7,69.2
Very High,979,12.409704,11.60,4.8,23.6


In [49]:
pd.crosstab(
    traffic_df["time_of_day"],
    traffic_df["traffic_density_level"]
)

traffic_density_level,High,Low,Medium,Very High
time_of_day,,,,
Afternoon,309,201,467,80
Evening Peak,531,86,233,547
Morning Peak,470,100,234,346
Night,38,231,121,6


In [50]:
traffic_df["start_area"].unique()

<StringArray>
[    'Vasant Kunj', 'Greater Kailash',       'Janakpuri',    'Punjabi Bagh',
          'Rohini', 'Noida Sector 18',     'IGI Airport',   'Chandni Chowk',
     'Mayur Vihar',           'Okhla',      'Model Town',          'Dwarka',
           'Saket',       'Hauz Khas',     'Nehru Place',           'AIIMS',
       'Pitampura', 'Connaught Place',  'Rajouri Garden',         'Kalkaji',
     'Preet Vihar',      'Karol Bagh',    'Lajpat Nagar',        'Shahdara',
     'Civil Lines']
Length: 25, dtype: str

In [51]:
traffic_density_map = {
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Very High": 4
}

traffic_df["Traffic_Density_Score"] = (
    traffic_df["traffic_density_level"]
    .map(traffic_density_map)
)

In [52]:
traffic_df.groupby("weather_condition")["Traffic_Density_Score"].mean()

weather_condition
Clear       2.680521
Fog         2.610706
Heatwave    2.740132
Rain        2.611018
Name: Traffic_Density_Score, dtype: float64

In [53]:
traffic_df.groupby("time_of_day")["Traffic_Density_Score"].mean()

time_of_day
Afternoon       2.253548
Evening Peak    3.101646
Morning Peak    2.923478
Night           1.542929
Name: Traffic_Density_Score, dtype: float64

In [54]:
start_df = traffic_df[
    ["start_area", "time_of_day", "Traffic_Density_Score"]
].rename(columns={"start_area": "Area"})
start_df

,Area,time_of_day,Traffic_Density_Score
0,Vasant Kunj,Night,1
1,Greater Kailash,Night,1
2,Janakpuri,Morning Peak,3
3,Punjabi Bagh,Night,1
4,Rohini,Afternoon,2
...,...,...,...
3995,Rajouri Garden,Morning Peak,4
3996,Rohini,Morning Peak,3
3997,Preet Vihar,Morning Peak,3
3998,Karol Bagh,Evening Peak,3


In [55]:
end_df = traffic_df[
    ["end_area", "time_of_day", "Traffic_Density_Score"]
].rename(columns={"end_area": "Area"})

In [59]:
area_df = pd.concat([start_df, end_df], ignore_index=True)
area_df

,Area,time_of_day,Traffic_Density_Score
0,Vasant Kunj,Night,1
1,Greater Kailash,Night,1
2,Janakpuri,Morning Peak,3
3,Punjabi Bagh,Night,1
4,Rohini,Afternoon,2
...,...,...,...
7995,Dwarka,Morning Peak,4
7996,Dwarka,Morning Peak,3
7997,Lajpat Nagar,Morning Peak,3
7998,Hauz Khas,Evening Peak,3


In [64]:
area_time_score = area_df.groupby(["Area", "time_of_day"])["Traffic_Density_Score"].agg(
    trips="count",
    avg_density="mean"
).reset_index()

In [65]:
area_time_score

,Area,time_of_day,trips,avg_density
0,AIIMS,Afternoon,83,2.168675
1,AIIMS,Evening Peak,120,2.983333
2,AIIMS,Morning Peak,94,3.085106
3,AIIMS,Night,32,1.656250
4,Chandni Chowk,Afternoon,91,2.186813
...,...,...,...,...
95,Shahdara,Night,26,1.653846
96,Vasant Kunj,Afternoon,90,2.300000
97,Vasant Kunj,Evening Peak,117,3.153846
98,Vasant Kunj,Morning Peak,105,2.895238


In [68]:
area_time_score["Traffic_Score"] = (
    area_time_score["avg_density"] - area_time_score["avg_density"].min()
) / (
    area_time_score["avg_density"].max() -
    area_time_score["avg_density"].min()
)
area_time_score

,Area,time_of_day,trips,avg_density,Traffic_Score
0,AIIMS,Afternoon,83,2.168675,0.458258
1,AIIMS,Evening Peak,120,2.983333,0.876401
2,AIIMS,Morning Peak,94,3.085106,0.928639
3,AIIMS,Night,32,1.656250,0.195243
4,Chandni Chowk,Afternoon,91,2.186813,0.467568
...,...,...,...,...,...
95,Shahdara,Night,26,1.653846,0.194010
96,Vasant Kunj,Afternoon,90,2.300000,0.525664
97,Vasant Kunj,Evening Peak,117,3.153846,0.963921
98,Vasant Kunj,Morning Peak,105,2.895238,0.831184


In [72]:
area_time_score["trips_norm"] = (
    area_time_score["trips"] - area_time_score["trips"].min()
) / (
    area_time_score["trips"].max() - area_time_score["trips"].min()
)
area_time_score.rename(
    columns={"Traffic_Score": "density_norm"},
    inplace=True
)

In [73]:
area_time_score["Traffic_Score"] = (
    0.7 * area_time_score["density_norm"] +
    0.3 * area_time_score["trips_norm"]
)

In [74]:
area_time_score.to_csv(
    "../data/processed/traffic_profile.csv",
    index=False
)